In [ ]:
# --- ENVIRONMENT SETUP ---\n!pip install pandas numpy yfinance matplotlib nselib tqdm python-dateutil plotly nbformat playwright\n!playwright install firefox

# Institutional Thematic Index: Transport, Shipping & Logistics
**Index Administrator:** Rules-Based Framework (Institutional Grade)

### I. Methodology Overview
This index follows professional indexing standards by applying a systematic framework for selection, weighting, and maintenance.

### II. Target Theme & Keywords
The index targets the Transportation, Maritime Shipping, and Global Logistics sectors. The current targeting logic uses:
*   **Keywords:** `transport OR shipping OR logistics OR marine`
*   **Targeting:** Identifying companies specializing in global supply chains, maritime trade, and freight transportation infrastructure.
*   _**Customizable:** This keyword set is always customizable to target different thematic niches._

### III. Selection & Weighting Rules
*   **Listing History:** Minimum **3-month** trading history required for entry.
*   **Liquidity (Average Daily Taded Value):** 3-month rolling ADTV of **₹4 Crore**.
*   **Weighting:** Pure Free-Float Market Capitalization weighted.
*   **Rebalancing:** Quarterly (Mar, Jun, Sep, Dec) with **continuity divisor adjustments**.
*   **Foundation:** Initialized at **1000 points** on Dec 31, 2022.
*   _**Customizable:** This selection and weighing rules are always customizable to get most flexible system._

### IV. Index Calculation Formula
The index is calculated using the following methodology:

$$Index_{Value} = \frac{\text{Free Float Market Cap}}{\text{Divisor}}$$

**Definitions:**
*   **Free Float Market Cap:**   $\sum (Price_{current} \times Shares_{total} \times \text{Public Holding \%})$
*   **Divisor (Foundation):**   $\frac{\text{FF Mcap}}{1000}$
*   **Divisor (Rebalancing):**   $Divisor_{prev} \times \left( \frac{\text{FF Mcap}_{after}}{\text{FF Mcap}_{before}} \right)$

## Table of Contents

### Chapter 1: Masterlist Construction (Data Acquisition)
Establishing the raw stock pool from NSE metadata.

### Chapter 2: Theme Definition & Enrichment
Establishing the candidate stock pool and syncing fundamental/price data.
*   **2.1 Shareholding Sync:** Fetching historical public holding factors via NSE API.
*   **2.2 Price & Volume Sync:** Synchronizing daily OHLCV data from Yahoo Finance.
*   **2.2.1 Share Capital:** Deriving total shares for market cap weighting.
*   **2.3 Data Enrichment:** Calculating Volume EMA and price cleaning.
*   **2.4 Stock Pool Validation:** Segregating Foundation vs. Delayed Entry stock pools.

### Chapter 3: Index Construction Roadmap (Validation)
*   **3.1 Timeline:** Generating quarterly rebalancing checkpoints.
*   **3.2 Liquidity Filter:** Applying the ₹6 Crore ADTV threshold.
*   **3.2.a Free Float:** Fetching public holding factors for selected stocks.
*   **3.3 Construction:** Establishing the top 5 constituents and initial divisor.
*   **3.4 Continuity:** Calculating the sequential divisor chain.

### Chapter 4: Index Candle Builder
*   **4.1 OHLC Construction:** Building daily price action and exporting CSV.

### Chapter 5: Performance Analysis
*   **5.0 Visualization:** Interactive TradingView-style charts.
*   **5.1 Metrics:** Annualized returns, Volatility, Sharpe, and Drawdown.

### Chapter 6: Benchmark Comparison
Side-by-side analysis against Nifty 50, 500, and Auto indices.

In [12]:
import requests
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import datetime
import time
import threading
import os
import nselib.capital_market as cm
from tqdm import tqdm
from dateutil.relativedelta import relativedelta
import warnings

# --- PLOTTING & WARNING CONFIGURATION ---
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 7)

# --- CORE INDEX TIMELINE ---
# We start fetching data early (Sept 2021) to allow indicators (EMA/ADTV) to warm up
START_DATE = pd.Timestamp("2021-09-30")

# The formal date the index begins its calculation
INDEX_START_DATE = pd.Timestamp("2021-12-31")

## 1. Masterlist Construction (Data Acquisition)
We make a masterlist by from the NSE equity database. This includes granular industry classifications will be useful to target thematic targeting.

In [ ]:
class NSEService:
    """
    Singleton service to manage NSE API sessions and rate-limiting.
    """
    _instance = None
    _lock = threading.Lock()

    def __init__(self):
        self.session = requests.Session()
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
            "Referer": "https://www.nseindia.com/"
        }
        self.cache = {}
        self.last_init = 0

    @classmethod
    def get_instance(cls):
        with cls._lock:
            if cls._instance is None:
                cls._instance = cls()
            return cls._instance

    def _ensure_session(self):
        """Establish or refresh the NSE session cookies."""
        now = time.time()
        if now - self.last_init > 300: # Refresh every 5 minutes
            try:
                self.session.get("https://www.nseindia.com/", headers=self.headers, timeout=10)
                self.last_init = now
            except Exception as e:
                print(f"Session refresh failed: {e}")

    def get_quote_equity(self, symbol: str):
        """Fetch metadata for a single ticker."""
        self._ensure_session()
        try:
            url = f"https://www.nseindia.com/api/quote-equity?symbol={symbol.replace(' ', '%20')}"
            res = self.session.get(url, headers=self.headers, timeout=10)
            return res.json() if res.status_code == 200 else None
        except:
            return None

nse_service = NSEService.get_instance()

def reconstruct_masterlist(limit=None):
    """
    Rebuilds the stock_mega_masterlist.csv by scraping NSE.
    Use limit=10 for testing without overwriting the full file.
    """
    masterlist_file = 'stock_mega_masterlist.csv'
    
    print("Fetching full equity list from nselib...")
    equities = cm.equity_list()
    symbols = equities['SYMBOL'].tolist()
    
    if limit:
        print(f"DEMO MODE: Fetching first {limit} symbols...")
        symbols = symbols[:limit]
    
    data = []
    for s in tqdm(symbols, desc="Fetching Detailed Industry Data"):
        q = nse_service.get_quote_equity(s)
        ind = q.get('industryInfo', {}) if q else {}
        
        data.append({
            "stock symbol": s,
            "macro": ind.get('macro'),
            "sector": ind.get('sector'),
            "industry": ind.get('industry'),
            "basic_industry": ind.get('basicIndustry')
        })
    
    df = pd.DataFrame(data)
    
    if limit is None:
        df.to_csv(masterlist_file, index=False)
        print(f"Full masterlist saved to {masterlist_file}")
    else:
        print("--- Demo results to show workability of the code (Not Saved) ---")
        display(df.head())

# Default to small limit to prevent accidental full-scrapes in testing it takes almost 10+ minutes
reconstruct_masterlist(limit=10)
# reconstruct_masterlist() # <- run this to run it in full mode for all stocks of NSE

Fetching full equity list from nselib...
DEMO MODE: Fetching first 10 symbols...


Fetching Detailed Industry Data: 100%|██████████| 10/10 [00:03<00:00,  3.00it/s]

--- Demo results to show workability of the code (Not Saved) ---


,stock symbol,macro,sector,industry,basic_industry
0,20MICRONS,Commodities,Metals & Mining,Minerals & Mining,Industrial Minerals
1,21STCENMGM,Financial Services,Financial Services,Capital Markets,Other Capital Market related Services
2,360ONE,Financial Services,Financial Services,Capital Markets,Stockbroking & Allied
3,3BBLACKBIO,Healthcare,Healthcare,Healthcare Services,Healthcare Service Provider
4,3IINFOLTD,Information Technology,Information Technology,IT - Software,Computers - Software & Consulting


## 2. Theme Definition & Stock Pool Construction
The 'Candidate Stock Pool' is formed by searching for thematic keywords across all classification columns.


In [ ]:
# Load the latest masterlist
df_master = pd.read_csv('stock_mega_masterlist.csv')

# ----- KEYWORDS SEARCH ---
keywords_any = ['shipping','transport'] # OR gate: Result must have at least one of these
keywords_all = [] # AND gate: Result must have ALL of these (empty = no must-have)
keywords_exclude = ['mutual fund','etf'] # NOT gate: Result must NOT have these keywords

def matches_theme(row):
    """Checks if a stock matches the thematic keywords across any meta-column."""
    search_text = " ".join([str(row[c]).lower() for c in df_master.columns]).lower()
    
    # 1. OR logic: Any of the keywords
    or_match = any(kw.lower() in search_text for kw in keywords_any) if keywords_any else True
    
    # 2. AND logic: All of the keywords
    and_match = all(kw.lower() in search_text for kw in keywords_all) if keywords_all else True
    
    # 3. EXCLUDE logic: None of the keywords
    not_match = not any(kw.lower() in search_text for kw in keywords_exclude) if keywords_exclude else True
    
    return or_match and and_match and not_match

# Filter the stock_pool
candidate_stock_pool = df_master[df_master.apply(matches_theme, axis=1)].copy()
candidate_symbols = candidate_stock_pool['stock symbol'].tolist()

print(f"Theme Alignment Check:")
print(f"Total Stocks Scanned: {len(df_master)}")
print(f"Candidate Stock Pool: {len(candidate_symbols)} stocks matching the new theme.")
candidate_stock_pool.head(10)

Theme Alignment Check:
Total Stocks Scanned: 2494
Candidate Stock Pool: 53 stocks matching the new theme.


,stock symbol,name,macro,sector,industry,basic_industry,total_shares
64,ALLCARGO,ALLCARGO LOGISTICS LTD,Services,Services,Transport Services,Logistics Solution Provider,1.497778e+09
141,ASPINWALL,ASPINWALL & CO LTD,Services,Services,Transport Services,Logistics Solution Provider,7.818288e+06
147,ATL,ALLCARGO TERMINALS LTD,Services,Services,Transport Infrastructure,Port & Port services,2.912948e+08
148,ATLANTAA,ATLANTAA LIMITED,Services,Services,Transport Infrastructure,Road Assets - Toll Annuity Hybrid-Annuity,8.236600e+07
163,AVG,AVG LOGISTICS LIMITED,Services,Services,Transport Services,Logistics Solution Provider,1.505772e+07
251,BLACKBUCK,BLACKBUCK LIMITED,Services,Services,Transport Services,Transport Related Services,1.820852e+08
421,DREAMFOLKS,DREAMFOLKS SERVICES LTD,Services,Services,Transport Infrastructure,Airport & Airport services,5.321152e+07
438,ECOSMOBLTY,ECOS (INDIA) MOB & HOSP L,Services,Services,Transport Services,Road Transport,6.000000e+07
483,ESSARSHPNG,ESSAR SHIPPING LTD,Services,Services,Transport Services,Shipping,2.070338e+08
531,GANESHBE,GANESH BENZOPLAST LIMITED,Energy,Oil Gas & Consumable Fuels,Oil,Oil Storage & Transportation,7.201716e+07


## 2.1 Precision Shareholding Sync (NSE API)
We fetch historical shareholding patterns using our standalone script to avoid Jupyter hangs and ensure a stable Playwright session.

In [18]:
import subprocess
import sys

# Trigger the external fetcher script
# This script uses Firefox/Chromium to safely scrape the NSE API for Free Float history
if candidate_symbols:
    print(f"Triggering background fetch for {len(candidate_symbols)} stocks...")
    
    # We call the script as a separate process to prevent browser hangs in the notebook
    process = subprocess.Popen(
        [sys.executable, "fetch_shareholding_nse.py"] + candidate_symbols, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True, 
        encoding='utf-8'
    )
    
    # Stream output to the notebook window
    for line in process.stdout:
        print(line, end="")
        
    process.wait()
    print("\nShareholding sync complete.")

# takes more than 40 sec to start and more to fetch
# takes around 22-27 sec for every fresh fetch

Triggering background fetch for 53 stocks...
Fetching shareholding for 53 stocks using original logic...

NSE Shareholding: 100%|██████████| 53/53 [10:02<00:00, 11.38s/it]
Done.

Shareholding sync complete.


## 2.2 Price & Volume Sync (Yahoo Finance)
Syncing daily OHLCV data to our local cache folder. We use Adjusted prices to account for Corporate Actions.

In [19]:
folder_ohlc = "ohlc"
if not os.path.exists(folder_ohlc):
    os.makedirs(folder_ohlc)

# Sync each symbol
for sym in tqdm(candidate_symbols, desc="Syncing Price History"):
    path = os.path.join(folder_ohlc, f"{sym}_ohlc.csv")
    ticker = sym + ".NS"
    
    # Only download if we don't have it locally
    if not os.path.exists(path):
        try:
            df = yf.download(ticker, start=START_DATE, progress=False, auto_adjust=True)
            if not df.empty:
                # Handle MultiIndex if present
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                df.to_csv(path)
        except Exception as e:
            print(f"Failed to download {sym}: {e}")

print("OHLCV Local Cache Synchronized.")

Syncing Price History: 100%|██████████| 53/53 [00:23<00:00,  2.26it/s]

OHLCV Local Cache Synchronized.


## 2.2.1 Share Capital & Historical Assumptions
We derive the **Total Number of Shares** for each stock to enable historical Market Cap reconstruction.

### The "Historical Constant" Assumption
For this backtest, we assume the **Total Number of Shares** is a constant value across the entire timeline.
*   **Why this works:** Since our OHLCV data is **Adjusted** for corporate actions (Splits, Bonuses, Mergers), the relative price movements are already accounted for.
*   **Limitation:** This method does not account for **Rights Issues** or **ESOPs**, where the share count increases without a proportional price adjustment. In a professional production environment, a 'Shares Outstanding' timeline would be used.

In [22]:
# --- DERIVE TOTAL SHARES & UPDATE MASTERLIST ---
df_master = pd.read_csv('stock_mega_masterlist.csv')

# Ensure 'total_shares' column exists
if 'total_shares' not in df_master.columns:
    df_master['total_shares'] = 0.0

print(f"Syncing Share Capital for {len(candidate_symbols)} stocks...")

for sym in tqdm(candidate_symbols, desc="Calculating Share Capital"):
    try:
        # 1. Fetch Current Market Cap
        ticker = yf.Ticker(sym + ".NS")
        current_mcap = ticker.info.get('marketCap')
        
        # 2. Get Last Price from our local cache
        path = os.path.join("ohlc", f"{sym}_ohlc.csv")
        if os.path.exists(path) and current_mcap:
            df_ohlc = pd.read_csv(path, index_col=0, parse_dates=True)
            last_price = df_ohlc['Close'].iloc[-1]
            
            if last_price > 0:
                # 3. Derive Number of Shares
                shares = current_mcap / last_price
                
                # 4. Update Masterlist
                df_master.loc[df_master['stock symbol'] == sym, 'total_shares'] = int(shares)
    except Exception as e:
        print(f"Error for {sym}: {e}")

# Save updated masterlist
df_master.to_csv('stock_mega_masterlist.csv', index=False)
print("Masterlist updated with Total Shares.")


Syncing Share Capital for 53 stocks...


Calculating Share Capital:   0%|          | 0/53 [00:00<?, ?it/s]

Calculating Share Capital: 100%|██████████| 53/53 [00:16<00:00,  3.15it/s]

Masterlist updated with Total Shares.


## 2.3 Data Enrichment: Volume EMA & Rounding
Calculating the rebalance-linked Volume EMA and cleaning price data for better analysis.

In [ ]:
# Configuration for Volume EMA Window
rebalance_period = 'quarterly' #<-------------- user decision
ema_window = {'quarterly': 63, 'monthly': 22, 'half_monthly': 11}.get(rebalance_period, 63)

print(f"Enriching data with {ema_window}-day Volume EMA and Rounding...")

for sym in tqdm(candidate_symbols, desc="Enriching Local Data"):
    path = os.path.join("ohlc", f"{sym}_ohlc.csv")
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            
            # 1. Round price columns to 2 decimals
            df[['Open','High','Low','Close']] = df[['Open','High','Low','Close']].round(2)
            
            # 2. Calculate EMA on Volume for liquidity trend analysis
            df[f'EMA_{ema_window}_Volume'] = df['Volume'].ewm(span=ema_window, adjust=False).mean()
            
            # 3. Clean and Save (Keeping all rows)
            df[f'EMA_{ema_window}_Volume'] = df[f'EMA_{ema_window}_Volume'].fillna(0).astype(int)
            df.to_csv(path)
        except:
            continue

print("Data enrichment complete.")

Enriching data with 63-day Volume EMA and Rounding...


Enriching Local Data: 100%|██████████| 53/53 [00:01<00:00, 37.23it/s]

Data enrichment complete.


## 2.4 Stock Pool Validation: Founding Member Pool vs. Delayed Entry Pool

This is a critical step for **mathematical integrity**. To avoid "look-ahead bias" (using data that wasn't available at the time), we must segregate our thematic stocks based on their listing dates relative to our **Index Start Date (Dec 31, 2022)**.

### I. The Founding Member Pool (Day 1 Constituents)
*   **Definition:** Stocks that were active and had tradeable data on or before Dec 31, 2022.
*   **Role:** These stocks form the "Founding Members" of the index. They will be used to calculate the initial index value starting at **1000 points**.

### II. The Delayed Entry Pool (Future Contenders)
*   **Definition:** Newer thematic listings (e.g., Hyundai, Ola Electric) that entered the market after the index inception.
*   **Role:** These stocks are placed in a 'waiting area.' They become eligible to compete for a seat in the index during future quarterly rebalancing cycles, provided they meet the **3-Month Listing Age** requirement.

In [24]:
# --- UNIVERSE VALIDATION: POOL SEGREGATION ---
founding_member_pool = []
delayed_entry_pool = []

print(f"Segregating stock_pool for Index Foundation Date: {INDEX_START_DATE.date()}")

for sym in candidate_symbols:
    path = os.path.join("ohlc", f"{sym}_ohlc.csv")
    if os.path.exists(path):
        try:
            df_v = pd.read_csv(path, index_col=0, parse_dates=True)
            if not df_v.empty:
                # Check if the stock's first trading day is on or before inception
                if df_v.index.min() <= INDEX_START_DATE:
                    founding_member_pool.append(sym)
                else:
                    delayed_entry_pool.append(sym)
        except:
            continue

print(f"\nValidation Summary:")
print(f"✅ Founding Member Pool (Day 1 Starters): {len(founding_member_pool)} stocks:{founding_member_pool}")
print(f"⏳ Delayed Entry Pool (Future Contenders): {len(delayed_entry_pool)} stocks:{delayed_entry_pool}")


Segregating stock_pool for Index Foundation Date: 2021-12-31

Validation Summary:
✅ Founding Member Pool (Day 1 Starters): 32 stocks:['ALLCARGO', 'ASPINWALL', 'ATLANTAA', 'AVG', 'ESSARSHPNG', 'GANESHBE', 'GAYAHWS', 'GICL', 'GLOBALVECT', 'GPPL', 'INDIGO', 'KERNEX', 'MAHLOG', 'NAVKARCORP', 'NECCLTD', 'NOIDATOLL', 'PATINTLOG', 'ACCURACY', 'RIIL', 'SEAMECLTD', 'SNOWMAN', 'SVLL', 'TCI', 'TCIEXP', 'TOTAL', 'VRLLOG', 'BLUEDART', 'CONCOR', 'GESHIP', 'GMRAIRPORT', 'SCI', 'ADANIPORTS']
⏳ Delayed Entry Pool (Future Contenders): 21 stocks:['ATL', 'BLACKBUCK', 'DREAMFOLKS', 'ECOSMOBLTY', 'GATEWAY', 'GLOTTIS', 'HILINFRA', 'INNOVISION', 'OBCL', 'OMFREIGHT', 'RITCO', 'SCILAL', 'SHADOWFAX', 'SHREEJISPG', 'TIGERLOGS', 'TRANSWORLD', 'TREL', 'WCIL', 'AEGISVOPAK', 'DELHIVERY', 'JSWINFRA']


## 3.0 Data Loading & Weighting Engine Setup

In [ ]:
# --- LOAD ALL DATA INTO MEMORY FOR THE BACKTEST ---
prices_list = []
volumes_list = []
info_dict = {}

print("Loading cached data into analysis environment...")

for sym in tqdm(candidate_symbols, desc="Loading Matrix"):
    path = os.path.join("ohlc", f"{sym}_ohlc.csv")
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        prices_list.append(df['Close'].rename(sym))
        volumes_list.append(df['Volume'].rename(sym))
        
        # Load Free Float Factor from our Shareholding Sync
        sh_path = os.path.join("shareholding", f"{sym}_shareholding.csv")
        ff = pd.read_csv(sh_path).iloc[0]['Public']/100.0 if os.path.exists(sh_path) else 0.5
        
        # Approximate current Market Cap from Ticker Info
        try:
            mcap = yf.Ticker(sym+".NS").info.get('marketCap')
            info_dict[sym] = {'mcap': mcap, 'ff': ff}
        except:
            info_dict[sym] = {'mcap': None, 'ff': ff}

# Create consolidated DataFrames
prices = pd.concat(prices_list, axis=1).dropna(axis=0, how='all')
volumes = pd.concat(volumes_list, axis=1).reindex(index=prices.index)
info_df = pd.DataFrame.from_dict(info_dict, orient='index')

print(f"Matrix Ready: {prices.shape[1]} stocks across {prices.shape[0]} trading days.")

## 3.1 Index Checkpoint Roadmap
Before executing the backtest, we generate a chronological roadmap of all **Rebalancing Checkpoints**. 

*   **Base Index Value:** Initialized at **1000** on Foundation.
*   **Divisor Column:** This represents the index divisor (Base Market Cap) which will be calculated dynamically during the backtest to maintain continuity.

In [25]:
# --- SELECTION & WEIGHTING ENGINE ---

def calculate_weights(mcap_values, ff_factors):
    """
    Calculates Free-Float Market Cap weights.
    W(i) = FF_MCAP(i) / Sum(FF_MCAP)
    """
    adj_mcap = mcap_values * ff_factors
    return adj_mcap / adj_mcap.sum()

def check_eligibility(symbol, ref_date):
    """
    Rule: Stock must have been listed for at least 3 months (90 days).
    """
    path = os.path.join("ohlc", f"{symbol}_ohlc.csv")
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        if not df.empty:
            listing_age = (ref_date - df.index.min()).days
            return listing_age >= 90 # 3 Months
    return False

print("Selection Logic & Weighting Functions Initialized.")

Selection Logic & Weighting Functions Initialized.


In [26]:
# --- GENERATE INDEX CHECKPOINT ROADMAP ---
checkpoint_dates = []
current_date = INDEX_START_DATE

# Define Step based on rebalance_period
if rebalance_period == 'quarterly':
    step = pd.DateOffset(months=3)
elif rebalance_period == 'monthly':
    step = pd.DateOffset(months=1)
else:
    step = pd.DateOffset(days=15)

# Generate list of dates until today
while current_date <= pd.Timestamp(datetime.datetime.now()):
    checkpoint_dates.append(current_date)
    current_date += step

roadmap_data = []

for idx, c_date in enumerate(checkpoint_dates):
    # Find stocks listed at least 3 months prior to this checkpoint
    eligible_at_t = [s for s in candidate_symbols if check_eligibility(s, c_date)]
    
    roadmap_data.append({
        'Checkpoint Date': c_date.date(),
        'Divisor': "", # 1000 for Foundation, blank for rest
        'Total Stock Count': len(eligible_at_t),
        'Total Stock List': ", ".join(eligible_at_t[:5]) + ("..." if len(eligible_at_t) > 5 else "")
    })

index_checkpoint_data = pd.DataFrame(roadmap_data)

print(f"Index Checkpoint Roadmap generated for {rebalance_period} frequency.")
display(index_checkpoint_data)


Index Checkpoint Roadmap generated for quarterly frequency.


,Checkpoint Date,Divisor,Total Stock Count,Total Stock List
0,2021-12-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG..."
1,2022-03-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG..."
2,2022-06-30,,33,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG..."
3,2022-09-30,,36,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG..."
4,2022-12-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS..."
5,2023-03-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS..."
6,2023-06-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS..."
7,2023-09-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS..."
8,2023-12-30,,39,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG..."
9,2024-03-30,,40,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG..."


## 3.2 Eligibility Criteria: Liquidity & Selection
We now perform a deep-dive into each checkpoint date to filter for **Liquidity (ADTV)**.

### The Selection Process:
1.  **Identify Candidates**: Start with the `Total Stock List` from Section 3.1.
2.  **Calculate EMA Traded Value**: For each stock, we look up its **EMA Volume** and **Close Price** on that specific checkpoint date.
3.  **Liquidity Threshold**: A stock is only **Selected** if its *EMA Traded Value > ₹10 Crore*.
4.  **Finalize Index**: These 'Selected' stocks represent the final constituents for that rebalancing period.

In [ ]:
# --- CALCULATE LIQUIDITY ELIGIBILITY (SECTION 3.2) ---
# FIX: Added 'As-Of' logic to handle weekends/holidays
LIQUIDITY_LIMIT = 10_000_000 * 4 # (₹Cr) <-------------- user decision
selected_counts = []
selected_lists = []

print(f"Applying Liquidity Filter (Threshold: ₹{LIQUIDITY_LIMIT/10_000_000} Cr)...")

for idx, row in index_checkpoint_data.iterrows():
    c_date = pd.Timestamp(row['Checkpoint Date'])
    # We take the full candidate list from the Masterlist
    # To avoid the '...' truncation from the previous step's display
    candidate_list = [s for s in candidate_symbols if check_eligibility(s, c_date)]
    
    selected_at_t = []
    
    for sym in candidate_list:
        path = os.path.join("ohlc", f"{sym}_ohlc.csv")
        if os.path.exists(path):
            try:
                df_ohlc = pd.read_csv(path, index_col=0, parse_dates=True)
                
                # --- AS-OF LOGIC ---
                # Find the latest available trading day ON or BEFORE the checkpoint
                available_dates = df_ohlc.index[df_ohlc.index <= c_date]
                
                if not available_dates.empty:
                    latest_trading_day = available_dates[-1]
                    day_data = df_ohlc.loc[latest_trading_day]
                    
                    # Find the EMA column
                    ema_col = [c for c in df_ohlc.columns if 'EMA' in c and 'Volume' in c][0]
                    
                    # Calculate EMA Traded Value (EMA Volume * Price)
                    ema_traded_val = day_data[ema_col] * day_data['Close']
                    
                    if ema_traded_val >= LIQUIDITY_LIMIT:
                        selected_at_t.append(sym)
            except:
                continue
                
    selected_counts.append(len(selected_at_t))
    selected_lists.append(", ".join(selected_at_t))

# Update the main Checkpoint DataFrame
index_checkpoint_data['Selected Stock Count'] = selected_counts
index_checkpoint_data['Selected Stock List'] = selected_lists

print("Liquidity selection complete (Holiday-adjusted).")
display(index_checkpoint_data)


Applying Liquidity Filter (Threshold: ₹4.0 Cr)...
Liquidity selection complete (Holiday-adjusted).


,Checkpoint Date,Divisor,Total Stock Count,Total Stock List,Selected Stock Count,Selected Stock List
0,2021-12-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI..."
1,2022-03-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI..."
2,2022-06-30,,33,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",15,"ALLCARGO, GPPL, INDIGO, MAHLOG, NAVKARCORP, RI..."
3,2022-09-30,,36,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",17,"ALLCARGO, GATEWAY, GPPL, INDIGO, MAHLOG, NAVKA..."
4,2022-12-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GPPL, INDIGO, MAHLOG, NA..."
5,2023-03-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GANESHBE, GPPL, INDIGO, ..."
6,2023-06-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",17,"ALLCARGO, DREAMFOLKS, GATEWAY, GPPL, INDIGO, M..."
7,2023-09-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",19,"ALLCARGO, DREAMFOLKS, GANESHBE, GATEWAY, GPPL,..."
8,2023-12-30,,39,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",23,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE..."
9,2024-03-30,,40,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",25,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE..."


## 3.2.a Free Float Factor Retrieval
We now fetch the **Public Holding %** (Free Float Factor) for all stocks that passed the liquidity test. 

Storing these as a dictionary for each checkpoint ensures that our final index weighting engine has all the 'investability' data ready in memory.

In [33]:
# --- FETCH FREE FLOAT FACTORS (SECTION 3.2.a) ---
ff_percent_column = []

print("Fetching Free Float Factors for selected stocks...")

for idx, row in index_checkpoint_data.iterrows():
    c_date = pd.Timestamp(row['Checkpoint Date'])
    selected_list = [s.strip() for s in row['Selected Stock List'].split(",") if s.strip()]
    
    ff_dict = {}
    
    for sym in selected_list:
        try:
            sh_path = os.path.join("shareholding", f"{sym}_shareholding.csv")
            if os.path.exists(sh_path):
                df_sh = pd.read_csv(sh_path)
                df_sh['Date'] = pd.to_datetime(df_sh['Date'])
                avail_sh = df_sh.loc[df_sh['Date'] <= c_date]
                
                if not avail_sh.empty:
                    # Get latest public % on or before checkpoint
                    latest_sh = avail_sh.sort_values('Date', ascending=False).iloc[0]
                    ff_dict[sym] = round(float(latest_sh['Public']) / 100.0, 4)
                else:
                    ff_dict[sym] = 0.5000 # Fallback
            else:
                ff_dict[sym] = 0.5000 # Fallback
        except:
            ff_dict[sym] = 0.5000
            
    ff_percent_column.append(ff_dict)

# Update the Roadmap
index_checkpoint_data['Selected Stocks FF%'] = ff_percent_column

print("Free Float lookup complete.")
display(index_checkpoint_data.head())


Fetching Free Float Factors for selected stocks...
Free Float lookup complete.


,Checkpoint Date,Divisor,Total Stock Count,Total Stock List,Selected Stock Count,Selected Stock List,Selected Stocks FF%
0,2021-12-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':..."
1,2022-03-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':..."
2,2022-06-30,,33,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",15,"ALLCARGO, GPPL, INDIGO, MAHLOG, NAVKARCORP, RI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':..."
3,2022-09-30,,36,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",17,"ALLCARGO, GATEWAY, GPPL, INDIGO, MAHLOG, NAVKA...","{'ALLCARGO': 0.3008, 'GATEWAY': 0.6788, 'GPPL'..."
4,2022-12-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GPPL, INDIGO, MAHLOG, NA...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.33, 'GPPL..."


## 3.3 Index Construction & Initial Divisor
We now finalize the index composition and establish the **Starting Divisor**.

### The Construction Process:
1.  **Free-Float Calculation**: For every stock that passed the liquidity test, we calculate its **Free-Float Market Cap** at that specific checkpoint.
    *   We use the **As-Of Price** $	imes$ **Masterlist Total Shares** $	imes$ **Latest Public Holding %**.
2.  **Ranking & Selection**: We rank the stocks by size and keep only the **Top N** (Index Length) members.
3.  **Base Divisor**: For the inception date (Day 1), we calculate the divisor needed to set the index value at **1000**.

In [34]:
# --- INDEX CONSTRUCTION & INITIAL DIVISOR (SECTION 3.3) ---
# UPDATED: Now uses pre-fetched FF% dictionary from Section 3.2.a
INDEX_LENGTH = 10
final_stock_lists = []
final_counts = []

df_m = pd.read_csv('stock_mega_masterlist.csv').set_index('stock symbol')

print(f"Constructing index with Index Length: {INDEX_LENGTH}...")

for idx, row in index_checkpoint_data.iterrows():
    c_date = pd.Timestamp(row['Checkpoint Date'])
    selected_candidates = [s.strip() for s in row['Selected Stock List'].split(",") if s.strip()]
    ff_lookup = row['Selected Stocks FF%'] # Dictionary from 3.2.a
    
    ff_mcap_data = []
    
    for sym in selected_candidates:
        try:
            # 1. Get As-Of Price
            ohlc_path = os.path.join("ohlc", f"{sym}_ohlc.csv")
            df_o = pd.read_csv(ohlc_path, index_col=0, parse_dates=True)
            avail_p = df_o.loc[df_o.index <= c_date]
            if avail_p.empty: continue
            price_t = avail_p['Close'].iloc[-1]
            
            # 2. Get Total Shares
            tot_shares = df_m.loc[sym, 'total_shares']
            
            # 3. Use Pre-fetched FF Factor
            ff_factor = ff_lookup.get(sym, 0.5)
            
            # 4. Calculate FF-Mcap
            ff_mcap = price_t * tot_shares * ff_factor
            ff_mcap_data.append({'symbol': sym, 'ff_mcap': ff_mcap})
            
        except:
            continue
            
    if ff_mcap_data:
        df_ff = pd.DataFrame(ff_mcap_data).sort_values('ff_mcap', ascending=False)
        top_n_stocks = df_ff.head(INDEX_LENGTH)['symbol'].tolist()
        
        if idx == 0:
            total_ff_mcap = df_ff.head(INDEX_LENGTH)['ff_mcap'].sum()
            foundation_divisor = total_ff_mcap / 1000
            index_checkpoint_data.loc[idx, 'Divisor'] = f"{foundation_divisor:,.2f}"
            
        final_stock_lists.append(", ".join(top_n_stocks))
        final_counts.append(len(top_n_stocks))
    else:
        final_stock_lists.append("")
        final_counts.append(0)

index_checkpoint_data['Index Length'] = final_counts
index_checkpoint_data['Index Stock List'] = final_stock_lists

print("Index construction complete.")
display(index_checkpoint_data)


Constructing index with Index Length: 10...
Index construction complete.


,Checkpoint Date,Divisor,Total Stock Count,Total Stock List,Selected Stock Count,Selected Stock List,Selected Stocks FF%,Index Length,Index Stock List
0,2021-12-31,"1,346,259,298.99",32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, ALLCAR..."
1,2022-03-31,,32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, BLUEDA..."
2,2022-06-30,,33,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",15,"ALLCARGO, GPPL, INDIGO, MAHLOG, NAVKARCORP, RI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, BLUEDA..."
3,2022-09-30,,36,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",17,"ALLCARGO, GATEWAY, GPPL, INDIGO, MAHLOG, NAVKA...","{'ALLCARGO': 0.3008, 'GATEWAY': 0.6788, 'GPPL'...",10,"ADANIPORTS, DELHIVERY, CONCOR, INDIGO, GMRAIRP..."
4,2022-12-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GPPL, INDIGO, MAHLOG, NA...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.33, 'GPPL...",10,"ADANIPORTS, DELHIVERY, CONCOR, INDIGO, GMRAIRP..."
5,2023-03-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GANESHBE, GPPL, INDIGO, ...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.33, 'GANE...",10,"ADANIPORTS, DELHIVERY, INDIGO, CONCOR, GMRAIRP..."
6,2023-06-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",17,"ALLCARGO, DREAMFOLKS, GATEWAY, GPPL, INDIGO, M...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.3398, 'GA...",10,"ADANIPORTS, INDIGO, DELHIVERY, CONCOR, GMRAIRP..."
7,2023-09-30,,37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",19,"ALLCARGO, DREAMFOLKS, GANESHBE, GATEWAY, GPPL,...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.3398, 'GA...",10,"ADANIPORTS, INDIGO, DELHIVERY, GMRAIRPORT, CON..."
8,2023-12-30,,39,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",23,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE...","{'ALLCARGO': 0.3008, 'ATL': 0.3008, 'AVG': 0.3...",10,"ADANIPORTS, INDIGO, GMRAIRPORT, DELHIVERY, CON..."
9,2024-03-30,,40,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",25,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE...","{'ALLCARGO': 0.3008, 'ATL': 0.3008, 'AVG': 0.4...",10,"ADANIPORTS, INDIGO, GMRAIRPORT, DELHIVERY, CON..."


## 3.4 The Divisor Chain: Maintaining Index Continuity
We now calculate the **Divisor** for all subsequent rebalancing checkpoints.

### The Continuity Logic:
To ensure the index level (points) remains unaffected by changes in constituents or weighting, we apply the **Divisor Adjustment Formula**:

$$ \text{New Divisor} = \text{Prev Divisor} \times \left( \frac{\text{Mcap}_{After}}{\text{Mcap}_{Before}} \right) $$

1.  **Before Update Mcap:** Value of the **Previous** index members at the **Current** price.
2.  **After Update Mcap:** Value of the **New** index members at the **Current** price.
3.  **Pricing:** If a rebalance falls on a holiday, we use the **Next Available Trading Day**'s closing price.

In [35]:
# --- THE DIVISOR CHAIN CALCULATION (SECTION 3.4) ---
print("Calculating Divisor Chain for continuity...")

# Utility function to get 'Next Available Price'
def get_price_after_or_on(sym, target_date):
    path = os.path.join("ohlc", f"{sym}_ohlc.csv")
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    # Filter for dates on or after checkpoint
    after_p = df.loc[df.index >= target_date]
    return after_p['Close'].iloc[0] if not after_p.empty else None

for idx in range(1, len(index_checkpoint_data)):
    c_date = pd.Timestamp(index_checkpoint_data.loc[idx, 'Checkpoint Date'])
    
    # 1. Get Previous State
    prev_divisor_str = index_checkpoint_data.loc[idx-1, 'Divisor'].replace(",", "")
    prev_divisor = float(prev_divisor_str)
    prev_stocks = [s.strip() for s in index_checkpoint_data.loc[idx-1, 'Index Stock List'].split(",")]
    prev_ff_lookup = index_checkpoint_data.loc[idx-1, 'Selected Stocks FF%']
    
    # 2. Get Current State (New Index)
    curr_stocks = [s.strip() for s in index_checkpoint_data.loc[idx, 'Index Stock List'].split(",")]
    curr_ff_lookup = index_checkpoint_data.loc[idx, 'Selected Stocks FF%']
    
    # 3. Calculate Mcaps at CURRENT Price
    mcap_before = 0
    mcap_after = 0
    
    # Calculate BEFORE (Old stocks at Current Price)
    for sym in prev_stocks:
        price = get_price_after_or_on(sym, c_date)
        shares = df_m.loc[sym, 'total_shares']
        ff = prev_ff_lookup.get(sym, 0.5)
        mcap_before += (price * shares * ff)
        
    # Calculate AFTER (New stocks at Current Price)
    for sym in curr_stocks:
        price = get_price_after_or_on(sym, c_date)
        shares = df_m.loc[sym, 'total_shares']
        ff = curr_ff_lookup.get(sym, 0.5)
        mcap_after += (price * shares * ff)
        
    # 4. Apply Adjustment Formula
    if mcap_before > 0:
        new_divisor = prev_divisor * (mcap_after / mcap_before)
        index_checkpoint_data.loc[idx, 'Divisor'] = f"{new_divisor:,.2f}"

print("Divisor Chain Calculation Complete.")
display(index_checkpoint_data)


Calculating Divisor Chain for continuity...
Divisor Chain Calculation Complete.


,Checkpoint Date,Divisor,Total Stock Count,Total Stock List,Selected Stock Count,Selected Stock List,Selected Stocks FF%,Index Length,Index Stock List
0,2021-12-31,"1,346,259,298.99",32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, ALLCAR..."
1,2022-03-31,"1,321,566,926.34",32,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",14,"ALLCARGO, GPPL, INDIGO, MAHLOG, RIIL, TCI, TCI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, BLUEDA..."
2,2022-06-30,"1,315,259,228.22",33,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",15,"ALLCARGO, GPPL, INDIGO, MAHLOG, NAVKARCORP, RI...","{'ALLCARGO': 0.3008, 'GPPL': 0.5599, 'INDIGO':...",10,"ADANIPORTS, CONCOR, INDIGO, GMRAIRPORT, BLUEDA..."
3,2022-09-30,"1,723,183,659.41",36,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, ESSARSHPNG...",17,"ALLCARGO, GATEWAY, GPPL, INDIGO, MAHLOG, NAVKA...","{'ALLCARGO': 0.3008, 'GATEWAY': 0.6788, 'GPPL'...",10,"ADANIPORTS, DELHIVERY, CONCOR, INDIGO, GMRAIRP..."
4,2022-12-30,"1,752,896,110.79",37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GPPL, INDIGO, MAHLOG, NA...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.33, 'GPPL...",10,"ADANIPORTS, DELHIVERY, CONCOR, INDIGO, GMRAIRP..."
5,2023-03-30,"1,743,475,532.12",37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",16,"ALLCARGO, DREAMFOLKS, GANESHBE, GPPL, INDIGO, ...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.33, 'GANE...",10,"ADANIPORTS, DELHIVERY, INDIGO, CONCOR, GMRAIRP..."
6,2023-06-30,"1,828,310,534.39",37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",17,"ALLCARGO, DREAMFOLKS, GATEWAY, GPPL, INDIGO, M...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.3398, 'GA...",10,"ADANIPORTS, INDIGO, DELHIVERY, CONCOR, GMRAIRP..."
7,2023-09-30,"1,821,231,385.57",37,"ALLCARGO, ASPINWALL, ATLANTAA, AVG, DREAMFOLKS...",19,"ALLCARGO, DREAMFOLKS, GANESHBE, GATEWAY, GPPL,...","{'ALLCARGO': 0.3008, 'DREAMFOLKS': 0.3398, 'GA...",10,"ADANIPORTS, INDIGO, DELHIVERY, GMRAIRPORT, CON..."
8,2023-12-30,"1,821,231,385.57",39,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",23,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE...","{'ALLCARGO': 0.3008, 'ATL': 0.3008, 'AVG': 0.3...",10,"ADANIPORTS, INDIGO, GMRAIRPORT, DELHIVERY, CON..."
9,2024-03-30,"1,833,842,735.60",40,"ALLCARGO, ASPINWALL, ATL, ATLANTAA, AVG...",25,"ALLCARGO, ATL, AVG, DREAMFOLKS, GANESHBE, GATE...","{'ALLCARGO': 0.3008, 'ATL': 0.3008, 'AVG': 0.4...",10,"ADANIPORTS, INDIGO, GMRAIRPORT, DELHIVERY, CON..."


## 4.0 Index Candle Builder: Day-by-Day Construction
We now build the historical 'Price Action' of the index. 

### The Chunk-Based Process:
We process the timeline in quarterly chunks using the **Index Roadmap** as our map. For every day between two rebalancing checkpoints, we calculate the index level using the constituents and divisor active during that period.

*   **Open/Close Only:** We focus on Open and Close prices as they are synchronized across the underlying stocks.
*   **The 'All-or-Nothing' Rule:** To ensure mathematical integrity, an index candle is only formed if **all 5 constituents** have valid trading data for that day. 
*   **Missing Days:** Days where one or more stocks are missing are omitted and audited at the end.

In [53]:
# --- DAY-BY-DAY INDEX CANDLE BUILDER (SECTION 4.0) ---
index_results = []
omitted_days_count = 0
total_trading_days = 0

print("Building Index Candles...")

# Ensure df_m is available and correctly indexed
df_m = pd.read_csv('stock_mega_masterlist.csv').set_index('stock symbol')

for idx in range(len(index_checkpoint_data)):
    # 1. Get Chunk Parameters
    start_date = pd.Timestamp(index_checkpoint_data.loc[idx, 'Checkpoint Date'])
    if idx < len(index_checkpoint_data) - 1:
        end_date = pd.Timestamp(index_checkpoint_data.loc[idx+1, 'Checkpoint Date']) - pd.Timedelta(days=1)
    else:
        end_date = pd.Timestamp(datetime.datetime.now())
        
    divisor_str = index_checkpoint_data.loc[idx, 'Divisor'].replace(",", "")
    if not divisor_str: continue # Skip if divisor not calculated
    divisor = float(divisor_str)
    
    index_stocks = [s.strip() for s in index_checkpoint_data.loc[idx, 'Index Stock List'].split(",") if s.strip()]
    ff_lookup = index_checkpoint_data.loc[idx, 'Selected Stocks FF%']
    
    if not index_stocks: continue

    # 2. Load and Sync OHLC Data for the 5 Stocks for this period
    stock_dfs = {}
    for sym in index_stocks:
        path = os.path.join("ohlc", f"{sym}_ohlc.csv")
        if os.path.exists(path):
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            stock_dfs[sym] = df.loc[start_date:end_date]
        
    # 3. Find Common Trading Dates (Intersection)
    common_dates = None
    for sym in index_stocks:
        if sym not in stock_dfs: continue
        if common_dates is None:
            common_dates = set(stock_dfs[sym].index)
        else:
            common_dates = common_dates.intersection(set(stock_dfs[sym].index))
            
    if not common_dates: continue
    
    # Iterate through each trading day in the period
    sorted_dates = sorted(list(common_dates))
    
    for d in sorted_dates:
        sum_open_mcap = 0
        sum_close_mcap = 0
        
        for sym in index_stocks:
            row_data = stock_dfs[sym].loc[d]
            shares = df_m.loc[sym, 'total_shares']
            ff = ff_lookup.get(sym, 0.5)
            
            sum_open_mcap += (row_data['Open'] * shares * ff)
            sum_close_mcap += (row_data['Close'] * shares * ff)
            
        index_results.append({
            'Date': d,
            'Open': round(sum_open_mcap / divisor, 2),
            'Close': round(sum_close_mcap / divisor, 2)
        })
        total_trading_days += 1

# Convert to final DataFrame
index_ohlc = pd.DataFrame(index_results).set_index('Date')

print(f"\nIndex Construction Complete!")
print(f"Total Index Trading Days: {total_trading_days}")
if not index_ohlc.empty:
    print(f"Start Value: {index_ohlc['Close'].iloc[0]}")
    print(f"End Value: {index_ohlc['Close'].iloc[-1]}")
    display(index_ohlc.head())


Building Index Candles...

Index Construction Complete!
Total Index Trading Days: 1062
Start Value: 1000.0
End Value: 1725.66


,Open,Close
Date,,
2021-12-31,994.42,1000.00
2022-01-03,1003.86,1011.44
2022-01-04,1017.60,1018.41
2022-01-05,1017.12,1021.63
2022-01-06,1013.66,1015.10


## 4.1 Index Export & OHLC Fabrication
We finalize the index data by fabricating the High/Low values and exporting the result.

*   **Fabrication:** Since we only have synchronized Open/Close data, we set High as `max(Open, Close)` and Low as `min(Open, Close)`. 
*   **Storage:** The final index history is saved to the root directory for use in charts or external analysis.

In [37]:
# --- EXPORT INDEX CANDLESTICKS (SECTION 4.1) ---
if not index_ohlc.empty:
    # Fabricate High and Low
    index_ohlc['High'] = index_ohlc[['Open', 'Close']].max(axis=1)
    index_ohlc['Low'] = index_ohlc[['Open', 'Close']].min(axis=1)
    
    # Reorder columns for standard OHLC format
    index_final = index_ohlc[['Open', 'High', 'Low', 'Close']]
    
    # Save to CSV in base directory
    output_path = 'index_candlestick.csv'
    index_final.to_csv(output_path)
    
    print(f"Success! Index history saved to: {output_path}")
    display(index_final.tail())
else:
    print("Error: No index data available to export.")


Success! Index history saved to: index_candlestick.csv


,Open,High,Low,Close
Date,,,,
2026-04-22,1729.53,1729.53,1725.30,1725.30
2026-04-23,1708.60,1709.43,1708.60,1709.43
2026-04-24,1712.33,1712.33,1693.91,1693.91
2026-04-27,1703.18,1726.70,1703.18,1726.70
2026-04-28,1722.09,1725.66,1722.09,1725.66


## 5.0 Interactive Index Visualizer
We now bring the index to life with an interactive candlestick chart.

*   **Navigation:** Use your mouse to zoom in/out and pan across the historical timeline.
*   **Discrete Timeline:** Weekends and holidays are automatically removed to show a continuous flow of price action.
*   **Analysis:** Hover over any candle to see the exact Open, High, Low, and Close values for that day.

In [66]:
# --- INTERACTIVE CANDLESTICK VISUALIZER (SECTION 5.0) ---
import plotly.graph_objects as go

# 1. Prepare Data
# Ensure index is reset to get 'Date' as a column for Plotly
plot_df = index_final.copy().reset_index()
plot_df['Date'] = plot_df['Date'].dt.strftime('%Y-%m-%d')

# 2. Create Candlestick Chart
fig = go.Figure(data=[go.Candlestick(
    x=plot_df['Date'],
    open=plot_df['Open'],
    high=plot_df['High'],
    low=plot_df['Low'],
    close=plot_df['Close'],
    name='Index OHLC'
)])

# 3. Configure Layout for 'TradingView' Style
fig.update_layout(
    title='Thematic Index: Historical Price Action (Base 1000)',
    yaxis_title='Index Points',
    xaxis_title='Trading Date',
    template='plotly_dark', # Premium dark theme
    xaxis_rangeslider_visible=True, # Allows quick navigation across years
    # FORCE DISCRETE X-AXIS: This removes gaps for weekends/holidays
    xaxis_type='category',
    height=700
)

# 4. Clean up the X-axis labels (show fewer labels to avoid clutter)
fig.update_xaxes(
    nticks=20,
    tickangle=-45
)

fig.show()


## 5.1 Performance Metrics Scorecard
We quantify the success of the index using institutional-grade risk/return metrics.

In [43]:
# --- PERFORMANCE METRICS SCORECARD (SECTION 5.1) ---
import numpy as np

# 1. Basic Returns
start_val = index_final['Close'].iloc[0]
end_val = index_final['Close'].iloc[-1]
total_return = (end_val / start_val) - 1

# 2. Annualized Return (CAGR)
days_count = len(index_final)
years_count = days_count / 252 # Assuming 252 trading days per year
annualized_return = (end_val / start_val) ** (1 / years_count) - 1

# 3. Volatility & Sharpe
daily_returns = index_final['Close'].pct_change().dropna()
annualized_vol = daily_returns.std() * np.sqrt(252)
sharpe_ratio = (annualized_return - 0.06) / annualized_vol # Assuming 6% Risk-Free Rate

# 4. Maximum Drawdown
rolling_max = index_final['Close'].cummax()
drawdowns = (index_final['Close'] / rolling_max) - 1
max_drawdown = drawdowns.min()

# 5. Average Turnover per Rebalance
# We compare the 'Index Stock List' between rows in the Roadmap
turnovers = []
for i in range(1, len(index_checkpoint_data)):
    prev_set = set([s.strip() for s in index_checkpoint_data.loc[i-1, 'Index Stock List'].split(",") if s.strip()])
    curr_set = set([s.strip() for s in index_checkpoint_data.loc[i, 'Index Stock List'].split(",") if s.strip()])
    
    if not curr_set: continue
    
    # Number of changed stocks / Total stocks
    change_count = len(curr_set - prev_set)
    turnovers.append(change_count / len(curr_set))

avg_turnover = np.mean(turnovers) if turnovers else 0

# 6. Create the Results Table
metrics_data = {
    'Metric': [
        'Total Return (%)',
        'Annualized Return (%)',
        'Annualized Volatility (%)',
        'Sharpe Ratio',
        'Maximum Drawdown (%)',
        'Avg. Turnover per Rebalance (%)'
    ],
    'Our Index': [
        f"{total_return*100:.2f}%",
        f"{annualized_return*100:.2f}%",
        f"{annualized_vol*100:.2f}%",
        f"{sharpe_ratio:.2f}",
        f"{max_drawdown*100:.2f}%",
        f"{avg_turnover*100:.2f}%"
    ]
}

performance_table = pd.DataFrame(metrics_data).set_index('Metric')

print("Performance Scorecard Generated.")
display(performance_table)


Performance Scorecard Generated.


,Our Index
Metric,
Total Return (%),72.57%
Annualized Return (%),13.82%
Annualized Volatility (%),25.03%
Sharpe Ratio,0.31
Maximum Drawdown (%),-34.30%
Avg. Turnover per Rebalance (%),6.47%


## 6.0 Performance Analysis vs. Benchmark
We now put the index to the ultimate test by comparing it against national benchmarks. 

### Comparison Logic:
*   **Rebasing:** We scale the benchmark price so that it starts at **1000** on the same day as our index.
*   **Metrics Correlation:** We calculate the same risk/return metrics for the benchmark to provide a true side-by-side performance analysis.

In [67]:
# --- BENCHMARK COMPARISON ENGINE (SECTION 6.0) ---
def run_benchmark_comparison(bench_ticker, bench_name):
    print(f"Comparing Index vs {bench_name} ({bench_ticker})...")
    
    # 1. Fetch Benchmark Data (Using confirmed working tickers)
    start_d = index_ohlc.index.min()
    end_d = index_ohlc.index.max()
    bench_data = yf.download(bench_ticker, start=start_d, end=end_d + pd.Timedelta(days=1), progress=False)
    
    if bench_data.empty:
        print(f"Error: Could not fetch data for {bench_ticker}")
        return

    # 2. Reindex and Rebase to 1000
    b_close = bench_data['Close'].copy()
    if isinstance(b_close, pd.DataFrame): b_close = b_close.iloc[:, 0]
    
    bench_aligned = b_close.reindex(index_ohlc.index).ffill()
    bench_rebased = 1000 * (bench_aligned / bench_aligned.iloc[0])
    
    # 3. Plot Comparison
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=index_ohlc.index, y=index_ohlc['Close'], name='Our Index', line=dict(color='#00ff00', width=2)))
    fig.add_trace(go.Scatter(x=index_ohlc.index, y=bench_rebased, name=bench_name, line=dict(color='#ff9900', width=2, dash='dot')))
    
    fig.update_layout(
        title=f'Performance Comparison: Our Index vs {bench_name}',
        yaxis_title='Index Points (Base 1000)',
        template='plotly_dark',
        height=500
    )
    fig.show()

    # 4. Calculate Benchmark Metrics
    b_start = float(bench_rebased.iloc[0])
    b_end = float(bench_rebased.iloc[-1])
    b_total_ret = (b_end / b_start) - 1
    
    days_c = len(bench_rebased)
    years_c = days_c / 252
    b_ann_ret = (b_end / b_start) ** (1 / years_c) - 1
    
    b_daily_ret = bench_rebased.pct_change().dropna()
    b_ann_vol = float(b_daily_ret.std() * np.sqrt(252))
    b_sharpe = float((b_ann_ret - 0.06) / b_ann_vol)
    
    b_rolling_max = bench_rebased.cummax()
    b_drawdowns = (bench_rebased / b_rolling_max) - 1
    b_max_dd = float(b_drawdowns.min())

    # 5. Build Comparison Table
    comp_data = {
        'Metric': ['Total Return (%)', 'Annualized Return (%)', 'Annualized Volatility (%)', 'Sharpe Ratio', 'Maximum Drawdown (%)', 'Avg. Turnover per Rebalance (%)'],
        'Our Index': performance_table['Our Index'].tolist(),
        'Benchmark': [
            f"{b_total_ret*100:.2f}%",
            f"{b_ann_ret*100:.2f}%",
            f"{b_ann_vol*100:.2f}%",
            f"{b_sharpe:.2f}",
            f"{b_max_dd*100:.2f}%",
            "N/A"
        ]
    }
    display(pd.DataFrame(comp_data).set_index('Metric'))


## 6.1 Run Comparisons

In [68]:
# --- COMPARE VS NIFTY 50 ---
run_benchmark_comparison('^NSEI', 'Nifty 50')

Comparing Index vs Nifty 50 (^NSEI)...


,Our Index,Benchmark
Metric,,
Total Return (%),72.57%,38.45%
Annualized Return (%),13.82%,8.03%
Annualized Volatility (%),25.03%,14.02%
Sharpe Ratio,0.31,0.14
Maximum Drawdown (%),-34.30%,-16.47%
Avg. Turnover per Rebalance (%),6.47%,N/A


In [69]:
# --- COMPARE VS NIFTY 500 ---
run_benchmark_comparison('^CRSLDX', 'Nifty 500')

Comparing Index vs Nifty 500 (^CRSLDX)...


,Our Index,Benchmark
Metric,,
Total Return (%),72.57%,51.93%
Annualized Return (%),13.82%,10.43%
Annualized Volatility (%),25.03%,14.74%
Sharpe Ratio,0.31,0.30
Maximum Drawdown (%),-34.30%,-18.84%
Avg. Turnover per Rebalance (%),6.47%,N/A


# Final Methodology Notes & Technical Limitations

### 1. Assumption on Share Capital & Corporate Actions
For this backtest, we have assumed that the **Total Number of Shares** for each stock remains fixed throughout the timeline. While this might seem like a significant simplification, it is largely mitigated by our use of **Adjusted Close Prices**. Since splits and bonuses reduce the price while increasing the share count proportionally, the resulting Market Cap remains consistent. However, this model does not currently account for capital changes like **Rights Issues** or **ESOPs**, where the Market Cap can shift by a certain percentage without a corresponding price adjustment. With more granular historical shareholding data, this refinement can be integrated in future versions.

### 2. Rebalancing Logic & Divisor Dependency
The rebalancing of the index (constituent changes or free-float updates) is performed using the **Close Prices** of the stocks on the rebalancing day. When the stock pool or public holding percentages change, the **Divisor** is updated to maintain continuity. In this backtest, we use this newly derived divisor to calculate both the Open and Close values for that specific day. Mathematically, this implies that the "Open" value has a dependency on the "Close" price of the same day (a future value). While this is a standard backtesting simplification, it differs from a live market environment where the divisor is typically adjusted at the previous day's close to be ready for the next day's open.

### 3. Resolution of Candles (Highs & Lows)
This index engine intentionally omits the calculation of **High** and **Low** values for the daily candlesticks. Because different stocks within the pool hit their respective highs and lows at different times during the trading session, simply aggregating the underlying highs and lows would result in a misleading and fabricated representation of the index's intraday range. To implement accurate High/Low tracking, the engine would require high-resolution (5-minute or 15-minute) tick data to synchronize the exact timestamps of price movements across all constituents.

### How to Run this Engine
If you wish to run this engine on your own machine trying all the customisaitions `[keyword filter, start date, index participant count, benchmark comparison, others]`:
1. Place this **.ipynb** file and the **fetch_shareholding_nse.py** script in the same folder.
2. Ensure all libraries are installed by running the **Environment Setup** cell at the very top of this notebook.
3. The code is designed to run independently and will automatically rebuild all data folders (OHLC and Shareholding) from scratch and perform all the run

_by Ranit Biswas_  
_mail: ranit.biswas@iitgn.ac.in_  
_website developed by me: https://AlgoXR.in_